# CricShot CNN-LSTM — Evaluation & Visualization Notebook

Sections:
1. Config & Model Definition
2. Load Models
3. Load Test Dataset
4. Full Evaluation (accuracy, precision, recall, F1, confusion matrix, per-class metrics)
5. Single-Sample Forward Pass Visualization (CNN activations, LSTM attention, softmax probs) with sliders
6. Gradient Flow Visualization


## Imports & Config

In [ ]:
!pip install scikit-learn ipywidgets

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import ipywidgets as widgets
from IPython.display import display, clear_output
import random
import warnings
warnings.filterwarnings('ignore')

# ── Paths — EDIT THESE ────────────────────────────────────────────────────────
PREPROCESSED_DIR   = './preprocessed_16_224_224'   # root of test/val/train split folders
BEST_MODEL_PATH    = './models/best_model.pth'
EARLIER_MODEL_PATH = './models/epoch_020.pth' # change to the checkpoint you want to compare

# ── Constants (must match training) ───────────────────────────────────────────
NUM_FRAMES   = 16
FRAME_H      = 224
FRAME_W      = 224
NUM_CLASSES  = 10
CNN_CHANNELS = [32, 64, 128, 256]
CNN_PROJ_DIM = 256
LSTM_HIDDEN  = 256
LSTM_LAYERS  = 1
CLF_HIDDEN   = 128
CLF_DROPOUT  = 0.5
BATCH_SIZE   = 16

CLASS_NAMES = ['cover', 'defense', 'flick', 'hook', 'late_cut',
               'lofted', 'pull', 'square_cut', 'straight', 'sweep']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 1, 1, 3)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 1, 1, 3)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'Classes: {CLASS_NAMES}')

## Model Definition

In [ ]:

class CNNEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        ch = CNN_CHANNELS

        self.features = nn.Sequential(
            # Block 1: 224 -> 112
            nn.Conv2d(3, ch[0], 3, padding=1),
            nn.BatchNorm2d(ch[0]), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 2: 112 -> 56
            nn.Conv2d(ch[0], ch[1], 3, padding=1),
            nn.BatchNorm2d(ch[1]), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            
            nn.Dropout2d(p=0.2),

            # Block 3: 56 -> 28
            nn.Conv2d(ch[1], ch[2], 3, padding=1),
            nn.BatchNorm2d(ch[2]), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 4: 28 -> 14
            nn.Conv2d(ch[2], ch[3], 3, padding=1),
            nn.BatchNorm2d(ch[3]), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            
            nn.Dropout2d(p=0.2)
        )
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.project = nn.Linear(ch[3] * 4*4, CNN_PROJ_DIM)

    def forward(self, x):
        x = self.features(x)

        x = self.pool(x).flatten(1)
        return self.project(x)

class LSTMEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=CNN_PROJ_DIM,
            hidden_size=LSTM_HIDDEN,
            num_layers=LSTM_LAYERS,
            batch_first=True,
            bidirectional=True,
        )

        self.dropout = nn.Dropout(p=0.3)
        self._init_weights()
        self.attn_score = nn.Linear(LSTM_HIDDEN * 2, 1)

    def _init_weights(self):
        for name, param in self.lstm.named_parameters():
            if 'weight_hh' in name:
                nn.init.orthogonal_(param)
            elif 'weight_ih' in name:
                nn.init.xavier_uniform_(param)
            elif 'bias' in name:
                nn.init.constant_(param, 0)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.dropout(out) 
        scores = self.attn_score(out)   # (B, T, 1)
        weights = torch.softmax(scores, dim=1)  # (B, T, 1)
        context = (out * weights).sum(dim=1)    # (B, H*2)        
        return context
        
class ClassifierHead(nn.Module):
    """
    Input  : (B, 256)
    Output : (B, num_classes)  logits
    """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(LSTM_HIDDEN*2, CLF_HIDDEN),
            nn.ReLU(),
            nn.Dropout(CLF_DROPOUT),
            nn.Linear(CLF_HIDDEN, NUM_CLASSES),
        )

    def forward(self, x):
        return self.net(x)
        

class CricketActionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn  = CNNEncoder()
        self.lstm = LSTMEncoder()
        self.head = ClassifierHead()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        B, T, C, H, W = x.shape
        feat = self.cnn(x.view(B * T, C, H, W)).view(B, T, -1)
        context = self.lstm(feat)
        return self.head(context)

print('Model classes defined.')

## Load Models

In [ ]:
def load_model(path, device=DEVICE):
    """Load a CricketActionNet from a checkpoint, handling DataParallel keys."""
    ckpt  = torch.load(path, map_location=device)
    state = ckpt.get('model_state', ckpt)

    # Strip 'module.' prefix added by DataParallel
    state = {k.replace('module.', ''): v for k, v in state.items()}

    model = CricketActionNet().to(device)
    model.load_state_dict(state)
    model.eval()
    epoch = ckpt.get('epoch', '?')
    print(f'Loaded: {Path(path).name}  (epoch={epoch}, val_acc={ckpt.get("val_acc", "?")}, val_loss={ckpt.get("val_loss", "?"):.4f})')
    return model


best_model    = load_model(BEST_MODEL_PATH)
earlier_model = load_model(EARLIER_MODEL_PATH)

print(f'\nParameters: {sum(p.numel() for p in best_model.parameters()):,}')

## Test Dataset & Loader

In [ ]:
from tqdm.notebook import tqdm

MEAN = IMAGENET_MEAN
STD  = IMAGENET_STD

class TestDataset(Dataset):
    """Loads test split into RAM as float16, casts to float32 on __getitem__."""
    def __init__(self, split='test'):
        self.samples = []
        base = Path(PREPROCESSED_DIR) / split
        all_paths = []
        for cls in CLASS_NAMES:
            cls_dir = base / cls
            if not cls_dir.exists(): continue
            for p in cls_dir.glob('*.npy'):
                all_paths.append((p, CLASS_TO_IDX[cls]))

        print(f'Loading {split} split → {len(all_paths)} clips into RAM...')
        for path, label in tqdm(all_paths, unit='clip'):
            frames = np.load(path).astype(np.float32) / 255.0
            frames = (frames - MEAN) / STD
            frames = frames.transpose(0, 3, 1, 2)   # (T, C, H, W)
            self.samples.append((torch.from_numpy(frames).half(), label, str(path)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        clip, label, path = self.samples[i]
        return clip.float(), label, path


test_dataset = TestDataset('test')
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=0)
print(f'Test batches: {len(test_loader)}')

## Full Evaluation (accuracy, precision, recall, F1, confusion matrix, per-class)

In [ ]:
@torch.no_grad()
def evaluate_model(model, loader, label='Model'):
    """Run full evaluation. Returns all_preds, all_labels."""
    model.eval()
    all_preds, all_labels = [], []

    for clips, labels, _ in tqdm(loader, desc=f'Eval [{label}]'):
        clips  = clips.to(DEVICE)
        logits = model(clips)
        preds  = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

    return np.array(all_preds), np.array(all_labels)


def print_metrics(preds, labels, model_label='Best Model'):
    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, average='macro', zero_division=0)
    rec  = recall_score(labels, preds, average='macro', zero_division=0)
    f1   = f1_score(labels, preds, average='macro', zero_division=0)

    print(f'\n{'═'*50}')
    print(f'  {model_label}')
    print(f'{'═'*50}')
    print(f'  Accuracy   : {acc*100:.2f}%')
    print(f'  Precision  : {prec*100:.2f}%  (macro)')
    print(f'  Recall     : {rec*100:.2f}%  (macro)')
    print(f'  F1 Score   : {f1*100:.2f}%  (macro)')
    print(f'{'─'*50}')
    print('\nPer-class Report:')
    print(classification_report(labels, preds, target_names=CLASS_NAMES, digits=3))
    return acc, prec, rec, f1


preds_best, labels_true = evaluate_model(best_model, test_loader, 'Best Model')
metrics_best = print_metrics(preds_best, labels_true, 'Best Model')

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
def plot_confusion_matrix(preds, labels, title='Confusion Matrix', normalize=True):
    cm = confusion_matrix(labels, preds)
    if normalize:
        cm_display = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        fmt, vmax  = '.2f', 1.0
    else:
        cm_display, fmt, vmax = cm, 'd', cm.max()

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    for ax, data, title_sfx, fmt_ in zip(
            axes,
            [cm.astype(float) / cm.sum(axis=1, keepdims=True), cm],
            ['Normalized', 'Raw Counts'],
            ['.2f', 'd']):
        sns.heatmap(data, annot=True, fmt=fmt_, cmap='Blues',
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                    linewidths=0.5, ax=ax,
                    vmin=0, vmax=1.0 if fmt_ == '.2f' else None)
        ax.set_title(f'{title} — {title_sfx}', fontsize=13, fontweight='bold')
        ax.set_xlabel('Predicted', fontsize=11)
        ax.set_ylabel('True', fontsize=11)
        ax.tick_params(axis='x', rotation=45)
        ax.tick_params(axis='y', rotation=0)

    plt.tight_layout()
    plt.savefig('confusion_matrix_best.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: confusion_matrix_best.png')


plot_confusion_matrix(preds_best, labels_true, 'Best Model')

In [ ]:
# ── Per-class bar chart ───────────────────────────────────────────────────────
def plot_per_class_metrics(preds, labels, title='Per-class Metrics'):
    p = precision_score(labels, preds, average=None, zero_division=0)
    r = recall_score(labels, preds, average=None, zero_division=0)
    f = f1_score(labels, preds, average=None, zero_division=0)

    x   = np.arange(NUM_CLASSES)
    w   = 0.25
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(x - w, p, w, label='Precision', color='#4C72B0')
    ax.bar(x,     r, w, label='Recall',    color='#DD8452')
    ax.bar(x + w, f, w, label='F1',        color='#55A868')
    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_NAMES, rotation=35, ha='right')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('per_class_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: per_class_metrics.png')


plot_per_class_metrics(preds_best, labels_true)

## Single-Sample Forward Pass Visualization with Sliders

Pick a correctly-predicted sample, then explore:
- CNN block activations (2 feature maps per block)
- LSTM attention weights over 16 frames
- Softmax probability bar chart

In [ ]:
# ── Hook-based activation + attention capturer ────────────────────────────────
class InspectableNet(nn.Module):
    """Wraps CricketActionNet to expose per-block activations and attention."""
    def __init__(self, base_model: CricketActionNet):
        super().__init__()
        self.cnn  = base_model.cnn
        self.lstm = base_model.lstm
        self.head = base_model.head
        self._activations = {}
        self._hooks = []
        self._register_hooks()

    def _register_hooks(self):
        for hook in self._hooks:
            hook.remove()
        self._hooks.clear()

        for block_name in ['block1', 'block2', 'block3', 'block4']:
            block = getattr(self.cnn, block_name)
            name  = block_name
            h = block.register_forward_hook(
                lambda mod, inp, out, n=name: self._activations.update({n: out.detach().cpu()})
            )
            self._hooks.append(h)

    def forward(self, x):
        """Returns logits, attention_weights."""
        self._activations.clear()
        B, T, C, H, W = x.shape
        feat           = self.cnn(x.view(B * T, C, H, W)).view(B, T, -1)
        context, attn  = self.lstm(feat)   # attn: (B, T, 1)
        logits         = self.head(context)
        return logits, attn

    @property
    def activations(self):
        return self._activations


inspectable = InspectableNet(best_model).to(DEVICE)
inspectable.eval()
print('Inspectable model ready.')

In [ ]:
# ── Find correctly-predicted samples ─────────────────────────────────────────
correct_indices = np.where(preds_best == labels_true)[0]
print(f'Correctly predicted: {len(correct_indices)} / {len(labels_true)}')

# Cache clips for the correct samples
correct_clips  = []
correct_labels = []
for idx in correct_indices:
    clip, label, _ = test_dataset[idx]
    correct_clips.append(clip)
    correct_labels.append(label)

print(f'Cached {len(correct_clips)} correct clips.')

In [ ]:
# ── Visualization helper functions ────────────────────────────────────────────

def unnormalize_frame(frame_tensor):
    """Convert normalized (C, H, W) tensor back to displayable (H, W, 3) uint8."""
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = frame_tensor.cpu().numpy().transpose(1, 2, 0)
    img  = (img * std + mean).clip(0, 1)
    return img


def plot_cnn_activations(activations, frame_idx, feat_map_a=0, feat_map_b=1):
    """Plot 2 feature maps from each of the 4 CNN blocks for a given frame."""
    block_names  = ['block1 (32ch, 112×112)', 'block2 (64ch, 56×56)',
                    'block3 (128ch, 28×28)',   'block4 (256ch, 14×14)']
    block_keys   = ['block1', 'block2', 'block3', 'block4']

    fig, axes = plt.subplots(4, 3, figsize=(13, 17))
    fig.suptitle(f'CNN Activations — Frame {frame_idx}', fontsize=14, fontweight='bold')

    for row, (key, name) in enumerate(zip(block_keys, block_names)):
        act = activations[key]   # (B*T, C, H, W)
        # frame_idx selects which of the 16 frames we look at
        frame_act = act[frame_idx]   # (C, H, W)

        n_channels = frame_act.shape[0]
        fa = feat_map_a % n_channels
        fb = feat_map_b % n_channels

        # Mean activation heatmap
        mean_act = frame_act.mean(0).numpy()
        axes[row, 0].imshow(mean_act, cmap='viridis')
        axes[row, 0].set_title(f'{name}\nMean activation', fontsize=9)
        axes[row, 0].axis('off')
        plt.colorbar(axes[row, 0].images[0], ax=axes[row, 0], fraction=0.046)

        # Feature map A
        axes[row, 1].imshow(frame_act[fa].numpy(), cmap='plasma')
        axes[row, 1].set_title(f'Feature map #{fa}', fontsize=9)
        axes[row, 1].axis('off')
        plt.colorbar(axes[row, 1].images[0], ax=axes[row, 1], fraction=0.046)

        # Feature map B
        axes[row, 2].imshow(frame_act[fb].numpy(), cmap='plasma')
        axes[row, 2].set_title(f'Feature map #{fb}', fontsize=9)
        axes[row, 2].axis('off')
        plt.colorbar(axes[row, 2].images[0], ax=axes[row, 2], fraction=0.046)

    plt.tight_layout()
    plt.savefig(f'cnn_activations_frame{frame_idx}.png', dpi=120, bbox_inches='tight')
    plt.show()


def plot_lstm_attention(attn_weights, true_label, pred_label, probs):
    """Bar chart of attention weights for each of the 16 frames."""
    weights = attn_weights[0, :, 0].cpu().numpy()   # (T,)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Attention weights
    colors = ['#e63946' if w == weights.max() else '#457b9d' for w in weights]
    axes[0].bar(range(NUM_FRAMES), weights, color=colors, edgecolor='white', linewidth=0.5)
    axes[0].set_xlabel('Frame Index')
    axes[0].set_ylabel('Attention Weight')
    axes[0].set_title(f'LSTM Attention Weights\nTrue: {CLASS_NAMES[true_label]}  |  Pred: {CLASS_NAMES[pred_label]}',
                      fontsize=11, fontweight='bold')
    axes[0].set_xticks(range(NUM_FRAMES))
    axes[0].grid(axis='y', alpha=0.3)

    # Softmax probabilities
    bar_colors = ['#2dc653' if i == pred_label else
                  '#e63946' if i == true_label else '#adb5bd'
                  for i in range(NUM_CLASSES)]
    axes[1].barh(CLASS_NAMES, probs, color=bar_colors, edgecolor='white')
    axes[1].set_xlabel('Probability')
    axes[1].set_title('Softmax Output Probabilities\n(green=pred, red=true if wrong)', fontsize=11, fontweight='bold')
    axes[1].set_xlim(0, 1)
    for i, v in enumerate(probs):
        axes[1].text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=8)
    axes[1].grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.savefig('lstm_attention_probs.png', dpi=120, bbox_inches='tight')
    plt.show()


def plot_input_frames(clip_tensor, title='Input Frames'):
    """Show all 16 frames of the input clip."""
    fig, axes = plt.subplots(2, 8, figsize=(18, 5))
    fig.suptitle(title, fontsize=12, fontweight='bold')
    for t, ax in enumerate(axes.flat):
        ax.imshow(unnormalize_frame(clip_tensor[t]))
        ax.set_title(f'Frame {t}', fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig('input_frames.png', dpi=100, bbox_inches='tight')
    plt.show()


print('Visualization helpers defined.')

In [ ]:
# ── Interactive Slider UI ─────────────────────────────────────────────────────
# Run one forward pass, then explore with sliders.

# Storage for last forward pass results
_cache = {}

def run_forward(sample_idx):
    """Run inspectable model on correct_clips[sample_idx], store results."""
    clip  = correct_clips[sample_idx].unsqueeze(0).to(DEVICE)   # (1, T, C, H, W)
    label = correct_labels[sample_idx]

    with torch.no_grad():
        logits, attn = inspectable(clip)

    probs = F.softmax(logits, dim=1)[0].cpu().numpy()
    pred  = probs.argmax()

    _cache['clip']        = correct_clips[sample_idx]   # (T, C, H, W)
    _cache['activations'] = dict(inspectable.activations)
    _cache['attn']        = attn
    _cache['probs']       = probs
    _cache['pred']        = int(pred)
    _cache['label']       = int(label)
    print(f'Sample {sample_idx} | True: {CLASS_NAMES[label]} | Pred: {CLASS_NAMES[pred]} | Conf: {probs[pred]:.3f}')


# ── Widgets ───────────────────────────────────────────────────────────────────
w_sample   = widgets.IntSlider(min=0, max=len(correct_clips)-1, value=0, step=1,
                               description='Sample #', continuous_update=False,
                               style={'description_width': '80px'}, layout=widgets.Layout(width='400px'))
w_frame    = widgets.IntSlider(min=0, max=NUM_FRAMES-1, value=0, step=1,
                               description='CNN Frame', continuous_update=False,
                               style={'description_width': '80px'}, layout=widgets.Layout(width='400px'))
w_fmap_a   = widgets.IntSlider(min=0, max=31, value=0, step=1,
                               description='Feat map A', continuous_update=False,
                               style={'description_width': '80px'}, layout=widgets.Layout(width='400px'))
w_fmap_b   = widgets.IntSlider(min=1, max=31, value=1, step=1,
                               description='Feat map B', continuous_update=False,
                               style={'description_width': '80px'}, layout=widgets.Layout(width='400px'))
w_show_btn = widgets.Button(description='🔄 Run Forward Pass', button_style='primary',
                             layout=widgets.Layout(width='200px'))
w_viz_btn  = widgets.Button(description='📊 Show All Visuals', button_style='success',
                             layout=widgets.Layout(width='200px'))
w_output   = widgets.Output()

def on_run_forward(b):
    with w_output:
        clear_output(wait=True)
        run_forward(w_sample.value)
        plot_input_frames(_cache['clip'],
                          f'Input — {CLASS_NAMES[_cache["label"]]} (pred: {CLASS_NAMES[_cache["pred"]]})')

def on_show_visuals(b):
    if 'activations' not in _cache:
        print('⚠️  Click "Run Forward Pass" first.')
        return
    with w_output:
        clear_output(wait=True)
        plot_input_frames(_cache['clip'],
                          f'Input — {CLASS_NAMES[_cache["label"]]} (pred: {CLASS_NAMES[_cache["pred"]]})')
        plot_cnn_activations(_cache['activations'],
                             frame_idx=w_frame.value,
                             feat_map_a=w_fmap_a.value,
                             feat_map_b=w_fmap_b.value)
        plot_lstm_attention(_cache['attn'],
                            _cache['label'], _cache['pred'], _cache['probs'])

w_show_btn.on_click(on_run_forward)
w_viz_btn.on_click(on_show_visuals)

controls = widgets.VBox([
    widgets.HTML('<b style="font-size:14px">🎛️ Visualization Controls</b>'),
    widgets.HTML('<i>1. Set sample, click Run. 2. Adjust frame/feature maps, click Show.</i>'),
    w_sample,
    widgets.HBox([w_show_btn, w_viz_btn]),
    widgets.HTML('<hr><b>CNN Activation Controls</b>'),
    w_frame,
    w_fmap_a,
    w_fmap_b,
    w_output,
])
display(controls)
# Auto-run first sample
run_forward(0)
plot_input_frames(_cache['clip'])

In [ ]:
# Show CNN activations for current cache (run after adjusting sliders)
plot_cnn_activations(_cache['activations'],
                     frame_idx=w_frame.value,
                     feat_map_a=w_fmap_a.value,
                     feat_map_b=w_fmap_b.value)

In [ ]:
# Show LSTM attention & softmax
plot_lstm_attention(_cache['attn'], _cache['label'], _cache['pred'], _cache['probs'])

## Gradient Flow Visualization

### What gradients tell us:
- **Gradient norm per layer** (backward from loss → each weight layer) shows how well the error signal flows
- **Vanishing gradients**: early layers get near-zero gradients → they barely learn
- **Exploding gradients**: large norms → instability
- **Comparing best vs. earlier**: if the earlier model has much lower early-layer gradients, this shows it hadn't learned to propagate signal efficiently — the best model's better gradient flow explains its higher accuracy
- **What evaluators learn from this**: that your training setup (CosineAnnealing + gradient clipping) helped maintain healthy flow, and that the best checkpoint represents a well-converged model

In [ ]:
def compute_gradient_flow(model, loader, device=DEVICE, n_batches=1):
    """
    Do one full forward+backward pass.
    Returns ordered list of (layer_name, grad_mean, grad_max) for all
    weight tensors with gradients.
    """
    model.train()   # need train mode for gradients to flow properly
    criterion = nn.CrossEntropyLoss()

    # zero grads
    for p in model.parameters():
        if p.grad is not None:
            p.grad.zero_()

    # take one batch
    clips, labels, _ = next(iter(loader))
    clips  = clips.to(device)
    labels = labels.to(device)

    logits = model(clips)
    loss   = criterion(logits, labels)
    loss.backward()

    model.eval()

    grad_data = []
    for name, param in model.named_parameters():
        if param.requires_grad and param.grad is not None and 'weight' in name:
            g     = param.grad.abs()
            grad_data.append({
                'name':     name,
                'mean':     g.mean().item(),
                'max':      g.max().item(),
                'std':      g.std().item(),
            })
    return loss.item(), grad_data


print('Computing gradient flow for BEST model...')
loss_best, grads_best = compute_gradient_flow(best_model, test_loader)
print(f'  Loss (best)   : {loss_best:.4f}')

print('Computing gradient flow for EARLIER model...')
loss_early, grads_early = compute_gradient_flow(earlier_model, test_loader)
print(f'  Loss (earlier): {loss_early:.4f}')

# Restore eval mode
best_model.eval()
earlier_model.eval()

In [ ]:
def plot_gradient_flow_comparison(grads_best, grads_early, loss_best, loss_early):
    """
    Side-by-side gradient norm plot for best vs. earlier model.
    Also plots mean gradient per layer overlaid for direct comparison.
    """
    # Shorten layer names for display
    def short_name(n):
        n = n.replace('cnn.block', 'CNN-B')
        n = n.replace('lstm.lstm', 'LSTM')
        n = n.replace('lstm.attn_score', 'Attn')
        n = n.replace('head.net', 'CLF')
        n = n.replace('.weight', '')
        return n

    names_best  = [short_name(g['name']) for g in grads_best]
    means_best  = [g['mean'] for g in grads_best]
    maxes_best  = [g['max']  for g in grads_best]

    names_early = [short_name(g['name']) for g in grads_early]
    means_early = [g['mean'] for g in grads_early]
    maxes_early = [g['max']  for g in grads_early]

    # Use best model's layer ordering as reference
    ref_names = names_best
    # Align earlier model to same order (may differ slightly)
    early_map  = {g['name']: g for g in grads_early}

    fig, axes = plt.subplots(3, 1, figsize=(16, 14))
    fig.suptitle('Gradient Flow — Best Model vs. Earlier Model\n'
                 f'(Best loss={loss_best:.4f}  |  Earlier loss={loss_early:.4f})',
                 fontsize=13, fontweight='bold')

    x  = np.arange(len(ref_names))
    w  = 0.38

    # ── Plot 1: Mean gradient per layer ──────────────────────────────────────
    axes[0].bar(x - w/2, means_best,  w, label=f'Best   (ep={best_model.state_dict})',
                color='#2196F3', alpha=0.85, edgecolor='white')
    # rebuild earlier means aligned to ref_names
    means_early_aligned = []
    for g in grads_best:
        match = next((e for e in grads_early if e['name'] == g['name']), None)
        means_early_aligned.append(match['mean'] if match else 0)

    axes[0].bar(x + w/2, means_early_aligned, w, label='Earlier',
                color='#FF5722', alpha=0.85, edgecolor='white')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(ref_names, rotation=60, ha='right', fontsize=8)
    axes[0].set_ylabel('Mean |Gradient|')
    axes[0].set_title('Mean Gradient Magnitude per Layer (higher = more learning signal)', fontsize=11)
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)

    # ── Plot 2: Max gradient per layer ────────────────────────────────────────
    maxes_early_aligned = []
    for g in grads_best:
        match = next((e for e in grads_early if e['name'] == g['name']), None)
        maxes_early_aligned.append(match['max'] if match else 0)

    axes[1].bar(x - w/2, maxes_best,          w, label='Best',    color='#2196F3', alpha=0.85, edgecolor='white')
    axes[1].bar(x + w/2, maxes_early_aligned, w, label='Earlier', color='#FF5722', alpha=0.85, edgecolor='white')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(ref_names, rotation=60, ha='right', fontsize=8)
    axes[1].set_ylabel('Max |Gradient|')
    axes[1].set_title('Max Gradient per Layer (watch for exploding gradients)', fontsize=11)
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.3)

    # ── Plot 3: Ratio (best / earlier) ────────────────────────────────────────
    ratios = [b / (e + 1e-12) for b, e in zip(means_best, means_early_aligned)]
    ratio_colors = ['#4CAF50' if r >= 1.0 else '#F44336' for r in ratios]
    axes[2].bar(x, ratios, color=ratio_colors, edgecolor='white')
    axes[2].axhline(1.0, color='black', linestyle='--', linewidth=1.5, label='Equal gradient')
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(ref_names, rotation=60, ha='right', fontsize=8)
    axes[2].set_ylabel('Ratio (Best / Earlier)')
    axes[2].set_title('Gradient Ratio — green means Best model has stronger gradient signal', fontsize=11)
    axes[2].legend()
    axes[2].grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('gradient_flow_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: gradient_flow_comparison.png')


plot_gradient_flow_comparison(grads_best, grads_early, loss_best, loss_early)

In [ ]:
# ── Gradient magnitude heatmap across layers ──────────────────────────────────
# Useful as a single compact slide image.

def plot_gradient_heatmap(grads_best, grads_early):
    def short_name(n):
        n = n.replace('cnn.block', 'CNN-B')
        n = n.replace('lstm.lstm', 'LSTM')
        n = n.replace('lstm.attn_score', 'Attn')
        n = n.replace('head.net', 'CLF')
        n = n.replace('.weight', '')
        return n

    layer_names = [short_name(g['name']) for g in grads_best]
    means_b     = np.array([g['mean'] for g in grads_best])

    means_early_aligned = []
    for g in grads_best:
        match = next((e for e in grads_early if e['name'] == g['name']), None)
        means_early_aligned.append(match['mean'] if match else 0)
    means_e = np.array(means_early_aligned)

    data = np.stack([means_b, means_e], axis=0)

    fig, ax = plt.subplots(figsize=(16, 3))
    im = ax.imshow(data, aspect='auto', cmap='YlOrRd')
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['Best Model', 'Earlier Model'], fontsize=11)
    ax.set_xticks(range(len(layer_names)))
    ax.set_xticklabels(layer_names, rotation=55, ha='right', fontsize=8)
    ax.set_title('Gradient Magnitude Heatmap (warmer = stronger gradient signal)', fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.02, pad=0.04)
    plt.tight_layout()
    plt.savefig('gradient_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: gradient_heatmap.png')


plot_gradient_heatmap(grads_best, grads_early)

## Gradient Statistics Table
Print a clean table of gradient stats for your presentation notes.

In [ ]:
import pandas as pd

def build_grad_table(grads_best, grads_early):
    rows = []
    early_map = {g['name']: g for g in grads_early}
    for g in grads_best:
        e = early_map.get(g['name'], {'mean': 0, 'max': 0, 'std': 0})
        rows.append({
            'Layer':         g['name'].replace('.weight',''),
            'Best Mean':     f"{g['mean']:.2e}",
            'Best Max':      f"{g['max']:.2e}",
            'Earlier Mean':  f"{e['mean']:.2e}",
            'Earlier Max':   f"{e['max']:.2e}",
            'Ratio B/E':     f"{g['mean'] / (e['mean']+1e-12):.2f}x",
        })
    return pd.DataFrame(rows)


df_grads = build_grad_table(grads_best, grads_early)
print('\nGradient Statistics (Best vs. Earlier):')
print(df_grads.to_string(index=False))

df_grads.to_csv('gradient_stats.csv', index=False)
print('\nSaved: gradient_stats.csv')

## Cell 9 — Summary of All Saved Outputs

| File | Contents |
|------|----------|
| `confusion_matrix_best.png` | Normalized + raw confusion matrix |
| `per_class_metrics.png` | Precision / Recall / F1 per class |
| `input_frames.png` | 16 frames of the selected clip |
| `cnn_activations_frame{N}.png` | Mean + 2 feature maps for each CNN block |
| `lstm_attention_probs.png` | Frame attention weights + softmax bar chart |
| `gradient_flow_comparison.png` | 3-panel gradient flow comparison |
| `gradient_heatmap.png` | Compact heatmap for slides |
| `gradient_stats.csv` | Numerical gradient table |